# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We will inspect the record sets and list their fields for reference, always using the `@id` for further steps.

> **Note:** All identifiers below reference the `@id` of each record set and field, per the Croissant standard.

In [ ]:
# List metadata about available record sets and fields
overview = {}
for rset in dataset.metadata.record_sets:
    print(f"RecordSet name: {rset.name}, @id: {rset.id}")
    field_ids = []
    for field in getattr(rset, 'fields', []):
        print(f"    Field: {getattr(field, 'name', None)}, @id: {field.id}, dataType: {getattr(field, 'data_type', None)}")
        field_ids.append(field.id)
    overview[rset.id] = field_ids
if not dataset.metadata.record_sets or len(dataset.metadata.record_sets) == 0:
    print("No named record sets: using all available records from main dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

The dataset may contain one default record set if the `record_sets` field is empty—this is often the case for simpler tabular datasets. If so, use `record_set=None` to extract all records.

In [ ]:
# Extract data from the main record set (or all records if empty)
record_sets = [rs.id for rs in getattr(dataset.metadata, 'record_sets', [])] if getattr(dataset.metadata, 'record_sets', []) else [None]
dataframes = {}

for rs_id in record_sets:
    print(f"Loading records from record_set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record_set {rs_id}")

# Show the columns for the first (or only) record set
main_rs_id = record_sets[0]
if main_rs_id in dataframes:
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print('No data loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping records. All field and record set references use their `@id` as specified by the Croissant schema.

Let's:
- Filter based on a numeric field (e.g., `MW_kDa` [Molecular Weight in kiloDaltons]) using its `@id` if found, otherwise another numeric field.
- Normalize that field.
- Optionally, group by a categorical field like `description` (protein name/description).

In [ ]:
# Try to select a numeric field (by inspecting columns)
df = dataframes.get(main_rs_id)

# Infer a likely numeric field (commonly protein mass, abundance, count)
numeric_candidates = [c for c in df.columns if any(substr in c.lower() for substr in ["abundance", "count", "mw", "coverage", "peptide", "number", "percent", "ratio", "score", "quantity"])]
if len(numeric_candidates) > 0:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0]

print(f"Selected numeric field for analysis: {numeric_field_id}")

# Basic filtering with an arbitrary threshold
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.1f} (mean): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} sample:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a categorical field (e.g., 'description' or other non-numeric field)
group_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and c != numeric_field_id]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())
else:
    group_field_id = None
    print('No suitable grouping field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot:
- Histogram of the selected numeric field
- Boxplot of normalized values
- Optional: Bar plot if a grouping field exists

`matplotlib` and `seaborn` are used for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f'Histogram of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x=filtered_df[f"{numeric_field_id}_normalized"])
plt.title(f'Boxplot of normalized {numeric_field_id} (filtered)')
plt.show()

if group_field_id:
    plt.figure(figsize=(10, 4))
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    sns.barplot(y=group_field_id, x=numeric_field_id, data=top_groups)
    plt.title(f'Top 10 {group_field_id} by mean {numeric_field_id}')
    plt.show()

## 6. Conclusion
We used the `mlcroissant` library to load and explore the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset.

- Loaded the metadata and automatically detected record sets and fields using their `@id`s.
- Loaded tabular data and performed basic filtering and normalization on a key numeric field.
- Created visualizations to inspect numeric distributions and category-grouped means.

**Next steps:** Consider further domain-specific processing, such as cluster analysis or integrating with protein ontologies for advanced bioinformatics exploration.